# C2 Selector Evaluation – Baselines, Ablation and Sensitivity

This notebook evaluates the **Component 2 fixed-budget review-evidence selector** against meaningful baselines and studies the contribution and sensitivity of its main design choices.

## Research Questions

1. Does the proposed selector provide more balanced evidence than usefulness-only selection?
2. Does redundancy control reduce repeated semantic evidence?
3. Does minority preservation improve supported-minority coverage?
4. What happens when individual selector components are removed?
5. How sensitive is the selector to the fixed review budget and provisional weights?

## Compared Methods

- Observed Helpful-Vote Top-K
- Predicted-Usefulness Top-K
- Helpfulness + MMR
- Aspect-Coverage
- Proposed without Minority
- Full Proposed

## Important Limitation

The final C1 aspect/sentiment output is not yet available.

Current aspect/sentiment and supported-minority fields are synthetic integration-test placeholders.

Therefore, aspect coverage, minority preservation and opinion-fidelity values are **prototype diagnostics only** and must not be treated as final research findings.

Observed helpful votes are used only as a retrospective reference baseline and are not predictor features in the learned usefulness model.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import jensenshannon

print("Selector evaluation environment ready")


Selector evaluation environment ready


## 1. Resolve Project Paths

This notebook reuses artifacts produced by the earlier C2 notebooks.

The main input is the selector-ready review table created by Notebook 05.


In [2]:
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "component_2").exists():

    COMPONENT_ROOT = (
        CURRENT_DIR
        / "component_2"
    )

elif CURRENT_DIR.name == "component_2":

    COMPONENT_ROOT = CURRENT_DIR

elif (
    CURRENT_DIR.name == "notebooks"
    and CURRENT_DIR.parent.name == "component_2"
):

    COMPONENT_ROOT = CURRENT_DIR.parent

else:

    raise RuntimeError(
        f"Could not locate component_2 from: {CURRENT_DIR}"
    )


PILOT_PATH = (
    COMPONENT_ROOT
    / "data"
    / "pilot"
    / "smartphone_pilot_cleaned.parquet"
)


INTEGRATION_DIR = (
    COMPONENT_ROOT
    / "results"
    / "integration"
)


SELECTOR_DIR = (
    COMPONENT_ROOT
    / "results"
    / "selector"
)


EVALUATION_DIR = (
    COMPONENT_ROOT
    / "results"
    / "selector_evaluation"
)


EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


SELECTOR_READY_PATH = (
    SELECTOR_DIR
    / "selector_ready_reviews.csv"
)


SAVED_FULL_PROPOSED_PATH = (
    SELECTOR_DIR
    / "selected_evidence_prototype.csv"
)


EMBEDDING_PATH = (
    INTEGRATION_DIR
    / "smartphone_review_embeddings.npy"
)


C1_MOCK_PATH = (
    INTEGRATION_DIR
    / "c1_mock_aspect_sentiment.csv"
)


SUPPORTED_MINORITY_PATH = (
    INTEGRATION_DIR
    / "mock_supported_minority_groups.csv"
)


required_paths = {

    "pilot":
        PILOT_PATH,

    "selector_ready":
        SELECTOR_READY_PATH,

    "saved_full_proposed":
        SAVED_FULL_PROPOSED_PATH,

    "embeddings":
        EMBEDDING_PATH,

    "mock_c1":
        C1_MOCK_PATH,

    "supported_minority":
        SUPPORTED_MINORITY_PATH,
}


print(
    "Current directory:",
    CURRENT_DIR
)

print(
    "Component root:",
    COMPONENT_ROOT
)


for name, path in (
    required_paths.items()
):

    print(
        f"{name}: {path.exists()} -> {path}"
    )


missing_paths = [

    str(path)

    for path
    in required_paths.values()

    if not path.exists()
]


if missing_paths:

    raise FileNotFoundError(

        "Missing required evaluation artifacts:\n"

        + "\n".join(
            missing_paths
        )
    )


print(
    "\nAll required evaluation artifacts found."
)


Current directory: d:\Research\Repository\J26-DS-349\component_2\notebooks
Component root: d:\Research\Repository\J26-DS-349\component_2
pilot: True -> d:\Research\Repository\J26-DS-349\component_2\data\pilot\smartphone_pilot_cleaned.parquet
selector_ready: True -> d:\Research\Repository\J26-DS-349\component_2\results\selector\selector_ready_reviews.csv
saved_full_proposed: True -> d:\Research\Repository\J26-DS-349\component_2\results\selector\selected_evidence_prototype.csv
embeddings: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\smartphone_review_embeddings.npy
mock_c1: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\c1_mock_aspect_sentiment.csv
supported_minority: True -> d:\Research\Repository\J26-DS-349\component_2\results\integration\mock_supported_minority_groups.csv

All required evaluation artifacts found.


## 2. Load and Validate Experiment Inputs

The selector-ready table is enriched with observed helpful-vote information from the cleaned smartphone pilot.

Observed helpful votes are used only for retrospective evaluation.


In [3]:
selector_ready_df = pd.read_csv(
    SELECTOR_READY_PATH
)


saved_full_proposed_df = pd.read_csv(
    SAVED_FULL_PROPOSED_PATH
)


reviews_df = pd.read_parquet(
    PILOT_PATH
)


embedding_matrix = np.load(
    EMBEDDING_PATH
)


c1_mock_df = pd.read_csv(
    C1_MOCK_PATH
)


supported_minority_df = pd.read_csv(
    SUPPORTED_MINORITY_PATH
)


if "log_helpful" not in reviews_df.columns:

    reviews_df[
        "log_helpful"
    ] = np.log1p(
        reviews_df[
            "helpful_vote"
        ].astype(float)
    )


observed_df = reviews_df[
    [
        "review_id",
        "helpful_vote",
        "log_helpful",
    ]
].copy()


experiment_df = (
    selector_ready_df.merge(

        observed_df,

        on="review_id",

        how="left",

        validate="one_to_one",
    )
)


print(
    "Candidate reviews:",
    len(experiment_df)
)


print(
    "Products:",
    experiment_df[
        "parent_asin"
    ].nunique()
)


print(
    "Embedding matrix:",
    embedding_matrix.shape
)


print(
    "Mock C1 rows:",
    len(c1_mock_df)
)


print(
    "Supported minority groups:",
    len(
        supported_minority_df
    )
)


Candidate reviews: 896
Products: 10
Embedding matrix: (896, 384)
Mock C1 rows: 1792
Supported minority groups: 111


In [4]:
assert experiment_df[
    "review_id"
].is_unique


assert experiment_df[
    "embedding_row"
].is_unique


assert experiment_df[
    "helpful_vote"
].notna().all()


assert experiment_df[
    "log_helpful"
].notna().all()


assert (
    embedding_matrix.shape[0]
    == len(experiment_df)
)


assert experiment_df[
    "usefulness_norm"
].between(
    0,
    1,
).all()


assert experiment_df[
    "representativeness_norm"
].between(
    0,
    1,
).all()


print(
    "Experiment input validation passed."
)


Experiment input validation passed.


## 3. Shared Experiment Helpers

The following functions are reused by all baselines, ablations and sensitivity experiments.


In [5]:
def parse_pipe_set(
    value
):

    if pd.isna(
        value
    ):

        return set()


    return {

        item

        for item
        in str(
            value
        ).split("|")

        if item
    }


c1_mock_df[
    "aspect_sentiment_key"
] = (

    c1_mock_df[
        "aspect_category"
    ].astype(str)

    + "::"

    + c1_mock_df[
        "sentiment"
    ].astype(str)
)


supported_minority_df[
    "aspect_sentiment_key"
] = (

    supported_minority_df[
        "aspect_category"
    ].astype(str)

    + "::"

    + supported_minority_df[
        "sentiment"
    ].astype(str)
)


supported_keys_by_product = (

    supported_minority_df

    .groupby(
        "parent_asin"
    )[
        "aspect_sentiment_key"
    ]

    .apply(set)

    .to_dict()
)


In [6]:
DEFAULT_PRIORITY_GROUPS = 2


def build_priority_minority_map(
    num_priority_groups=
        DEFAULT_PRIORITY_GROUPS,
):

    priority_map = {}


    for product_id, group in (

        supported_minority_df

        .groupby(
            "parent_asin"
        )
    ):

        sorted_group = (

            group

            .sort_values(

                [
                    "prevalence",
                    "support_count",
                    "avg_confidence",
                ],

                ascending=[
                    True,
                    False,
                    False,
                ],
            )

            .drop_duplicates(
                subset=[
                    "aspect_sentiment_key"
                ]
            )
        )


        priority_keys = (

            sorted_group[
                "aspect_sentiment_key"
            ]

            .head(
                num_priority_groups
            )

            .tolist()
        )


        priority_map[
            product_id
        ] = set(
            priority_keys
        )


    return priority_map


priority_minority_by_product = (
    build_priority_minority_map()
)


print(
    "Priority minority map created for",
    len(
        priority_minority_by_product
    ),
    "products."
)


Priority minority map created for 10 products.


In [7]:
def candidate_redundancy(
    candidate_embedding_row,
    selected_embedding_rows,
):

    if len(
        selected_embedding_rows
    ) == 0:

        return 0.0


    candidate_vector = (

        embedding_matrix[

            int(
                candidate_embedding_row
            )
        ]

        .reshape(
            1,
            -1,
        )
    )


    selected_vectors = (

        embedding_matrix[

            np.asarray(
                selected_embedding_rows,
                dtype=int,
            )
        ]
    )


    similarities = (

        cosine_similarity(

            candidate_vector,

            selected_vectors,
        )[0]
    )


    return max(

        0.0,

        float(
            np.max(
                similarities
            )
        ),
    )


In [8]:
def mean_pairwise_similarity(
    embedding_rows
):

    embedding_rows = list(
        embedding_rows
    )


    if len(
        embedding_rows
    ) < 2:

        return 0.0


    vectors = (

        embedding_matrix[

            np.asarray(
                embedding_rows,
                dtype=int,
            )
        ]
    )


    similarity_matrix = (

        cosine_similarity(
            vectors
        )
    )


    upper = np.triu_indices(

        len(
            embedding_rows
        ),

        k=1,
    )


    return float(

        np.mean(

            similarity_matrix[
                upper
            ]
        )
    )


In [9]:
def product_opinion_jsd(
    product_id,
    selected_review_ids,
):

    full_product_review_ids = set(

        experiment_df[

            experiment_df[
                "parent_asin"
            ] == product_id

        ][
            "review_id"
        ]
    )


    full_c1 = (

        c1_mock_df[

            c1_mock_df[
                "review_id"
            ].isin(
                full_product_review_ids
            )
        ]
    )


    selected_c1 = (

        c1_mock_df[

            c1_mock_df[
                "review_id"
            ].isin(

                set(
                    selected_review_ids
                )
            )
        ]
    )


    categories = sorted(

        set(
            full_c1[
                "aspect_sentiment_key"
            ]
        )

        |

        set(
            selected_c1[
                "aspect_sentiment_key"
            ]
        )
    )


    if not categories:

        return np.nan


    full_counts = (

        full_c1[
            "aspect_sentiment_key"
        ]

        .value_counts()
    )


    selected_counts = (

        selected_c1[
            "aspect_sentiment_key"
        ]

        .value_counts()
    )


    p = np.array(

        [

            full_counts.get(
                key,
                0,
            )

            for key
            in categories
        ],

        dtype=float,
    )


    q = np.array(

        [

            selected_counts.get(
                key,
                0,
            )

            for key
            in categories
        ],

        dtype=float,
    )


    if (
        p.sum() <= 0
        or q.sum() <= 0
    ):

        return np.nan


    p = (
        p
        / p.sum()
    )


    q = (
        q
        / q.sum()
    )


    js_distance = (

        jensenshannon(

            p,
            q,

            base=2.0,
        )
    )


    return float(
        js_distance ** 2
    )


In [10]:
def summarize_selection(
    selection_df,
    selector_name,
):

    records = []


    for product_id, full_group in (

        experiment_df

        .groupby(
            "parent_asin"
        )
    ):

        selected_group = (

            selection_df[

                selection_df[
                    "parent_asin"
                ] == product_id
            ]
        )


        all_aspects = set()


        for value in (

            full_group[
                "aspect_categories"
            ]
        ):

            all_aspects |= (

                parse_pipe_set(
                    value
                )
            )


        selected_aspects = set()


        for value in (

            selected_group[
                "aspect_categories"
            ]
        ):

            selected_aspects |= (

                parse_pipe_set(
                    value
                )
            )


        all_supported_groups = (

            supported_keys_by_product

            .get(
                product_id,
                set(),
            )
        )


        selected_supported_groups = set()


        for value in (

            selected_group[
                "supported_minority_groups"
            ]
        ):

            selected_supported_groups |= (

                parse_pipe_set(
                    value
                )
            )


        priority_groups = (

            priority_minority_by_product

            .get(
                product_id,
                set(),
            )
        )


        covered_priority = (

            selected_supported_groups

            & priority_groups
        )


        aspect_coverage = (

            len(
                selected_aspects
            )

            /

            max(
                len(
                    all_aspects
                ),
                1,
            )
        )


        if all_supported_groups:

            supported_minority_recall = (

                len(
                    selected_supported_groups
                    & all_supported_groups
                )

                /

                len(
                    all_supported_groups
                )
            )

        else:

            supported_minority_recall = (
                np.nan
            )


        if priority_groups:

            priority_minority_coverage = (

                len(
                    covered_priority
                )

                /

                len(
                    priority_groups
                )
            )

        else:

            priority_minority_coverage = (
                np.nan
            )


        records.append(

            {

                "selector":
                    selector_name,

                "parent_asin":
                    product_id,

                "selected_count":
                    len(
                        selected_group
                    ),

                "mean_usefulness_norm":
                    float(
                        selected_group[
                            "usefulness_norm"
                        ].mean()
                    ),

                "mean_representativeness_norm":
                    float(
                        selected_group[
                            "representativeness_norm"
                        ].mean()
                    ),

                "mean_observed_log_helpful":
                    float(
                        selected_group[
                            "log_helpful"
                        ].mean()
                    ),

                "aspect_coverage":
                    float(
                        aspect_coverage
                    ),

                "supported_minority_recall":
                    float(
                        supported_minority_recall
                    )
                    if not np.isnan(
                        supported_minority_recall
                    )
                    else np.nan,

                "priority_minority_coverage":
                    float(
                        priority_minority_coverage
                    )
                    if not np.isnan(
                        priority_minority_coverage
                    )
                    else np.nan,

                "mean_pairwise_similarity":
                    float(
                        mean_pairwise_similarity(

                            selected_group[
                                "embedding_row"
                            ].tolist()
                        )
                    ),

                "opinion_js_divergence":
                    float(
                        product_opinion_jsd(

                            product_id,

                            selected_group[
                                "review_id"
                            ].tolist(),
                        )
                    ),
            }
        )


    return pd.DataFrame(
        records
    )


In [11]:
def overall_summary(
    product_metrics_df
):

    return (

        product_metrics_df

        .groupby(
            "selector",
            as_index=False,
        )

        .agg(

            mean_usefulness_norm=(
                "mean_usefulness_norm",
                "mean",
            ),

            mean_representativeness_norm=(
                "mean_representativeness_norm",
                "mean",
            ),

            mean_observed_log_helpful=(
                "mean_observed_log_helpful",
                "mean",
            ),

            mean_aspect_coverage=(
                "aspect_coverage",
                "mean",
            ),

            mean_supported_minority_recall=(
                "supported_minority_recall",
                "mean",
            ),

            mean_priority_minority_coverage=(
                "priority_minority_coverage",
                "mean",
            ),

            mean_pairwise_similarity=(
                "mean_pairwise_similarity",
                "mean",
            ),

            mean_opinion_js_divergence=(
                "opinion_js_divergence",
                "mean",
            ),
        )
    )


## 4. Main Experiment Configuration

The same fixed budget is used for all main selector comparisons.

The initial prototype budget remains `k = 5`.


In [12]:
SELECT_K = 5


MMR_LAMBDA = 0.70


FULL_WEIGHTS = {

    "usefulness":
        0.30,

    "representativeness":
        0.25,

    "coverage":
        0.20,

    "minority":
        0.25,

    "redundancy":
        0.20,
}


MINORITY_RESERVED_SLOTS = 2


print(
    "Selection budget:",
    SELECT_K
)


print(
    "MMR lambda:",
    MMR_LAMBDA
)


print(
    "Full proposed weights:",
    FULL_WEIGHTS
)


Selection budget: 5
MMR lambda: 0.7
Full proposed weights: {'usefulness': 0.3, 'representativeness': 0.25, 'coverage': 0.2, 'minority': 0.25, 'redundancy': 0.2}


## 5. Observed Helpful-Vote Top-K

This retrospective reference baseline selects reviews with the highest observed helpful-vote counts.

It is not a deployable prediction method because the final vote count is only known after user interaction.


In [13]:
observed_frames = []


for product_id, group in (

    experiment_df

    .groupby(
        "parent_asin"
    )
):

    selected = (

        group

        .sort_values(

            [
                "helpful_vote",
                "log_helpful",
                "review_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],
        )

        .head(
            SELECT_K
        )

        .copy()
    )


    observed_frames.append(
        selected
    )


observed_helpful_topk_df = (

    pd.concat(
        observed_frames,
        ignore_index=True,
    )
)


print(
    "Observed Helpful-Vote Top-K rows:",
    len(
        observed_helpful_topk_df
    )
)


Observed Helpful-Vote Top-K rows: 50


## 6. Predicted-Usefulness Top-K

This baseline selects reviews with the highest Function 1 predicted usefulness.


In [14]:
predicted_frames = []


for product_id, group in (

    experiment_df

    .groupby(
        "parent_asin"
    )
):

    selected = (

        group

        .sort_values(

            [
                "usefulness_norm",
                "predicted_usefulness",
                "review_id",
            ],

            ascending=[
                False,
                False,
                True,
            ],
        )

        .head(
            SELECT_K
        )

        .copy()
    )


    predicted_frames.append(
        selected
    )


predicted_usefulness_topk_df = (

    pd.concat(
        predicted_frames,
        ignore_index=True,
    )
)


print(
    "Predicted-Usefulness Top-K rows:",
    len(
        predicted_usefulness_topk_df
    )
)


Predicted-Usefulness Top-K rows: 50


## 7. Helpfulness + MMR

Maximal Marginal Relevance balances usefulness with semantic novelty.

`MMR = λ × usefulness − (1 − λ) × redundancy`

The initial prototype uses `λ = 0.70`.


In [15]:
def select_mmr(
    product_df,
    budget,
    lambda_value=0.70,
):

    product_df = (

        product_df

        .copy()

        .reset_index(
            drop=True
        )
    )


    selected_indices = []

    selected_embedding_rows = []


    target_count = min(
        budget,
        len(
            product_df
        ),
    )


    while (
        len(
            selected_indices
        )
        < target_count
    ):

        candidates = []


        for idx, row in (

            product_df.iterrows()
        ):

            if idx in selected_indices:

                continue


            redundancy = (

                candidate_redundancy(

                    row[
                        "embedding_row"
                    ],

                    selected_embedding_rows,
                )
            )


            mmr_score = (

                lambda_value

                * float(
                    row[
                        "usefulness_norm"
                    ]
                )

                -

                (
                    1.0
                    - lambda_value
                )

                * redundancy
            )


            candidates.append(

                {

                    "idx":
                        idx,

                    "score":
                        float(
                            mmr_score
                        ),

                    "tie_usefulness":
                        float(
                            row[
                                "usefulness_norm"
                            ]
                        ),
                }
            )


        best = max(

            candidates,

            key=lambda item: (

                item[
                    "score"
                ],

                item[
                    "tie_usefulness"
                ],
            ),
        )


        best_idx = (
            best[
                "idx"
            ]
        )


        selected_indices.append(
            best_idx
        )


        selected_embedding_rows.append(

            int(

                product_df.loc[

                    best_idx,

                    "embedding_row",
                ]
            )
        )


    return (

        product_df.loc[
            selected_indices
        ]

        .copy()

        .reset_index(
            drop=True
        )
    )


In [16]:
mmr_frames = []


for product_id, group in (

    experiment_df

    .groupby(
        "parent_asin"
    )
):

    selected = select_mmr(

        product_df=group,

        budget=SELECT_K,

        lambda_value=
            MMR_LAMBDA,
    )


    mmr_frames.append(
        selected
    )


helpfulness_mmr_df = (

    pd.concat(
        mmr_frames,
        ignore_index=True,
    )
)


print(
    "Helpfulness + MMR rows:",
    len(
        helpfulness_mmr_df
    )
)


Helpfulness + MMR rows: 50


## 8. Aspect-Coverage Baseline

This baseline greedily maximizes newly introduced aspect categories.

Predicted usefulness is used as a tie-breaker.


In [17]:
def select_aspect_coverage(
    product_df,
    budget,
):

    product_df = (

        product_df

        .copy()

        .reset_index(
            drop=True
        )
    )


    selected_indices = []

    covered_aspects = set()


    target_count = min(

        budget,

        len(
            product_df
        ),
    )


    while (
        len(
            selected_indices
        )
        < target_count
    ):

        candidates = []


        for idx, row in (

            product_df.iterrows()
        ):

            if idx in selected_indices:

                continue


            candidate_aspects = (

                parse_pipe_set(

                    row[
                        "aspect_categories"
                    ]
                )
            )


            new_aspects = (

                candidate_aspects

                - covered_aspects
            )


            candidates.append(

                {

                    "idx":
                        idx,

                    "new_aspects":
                        len(
                            new_aspects
                        ),

                    "usefulness":
                        float(
                            row[
                                "usefulness_norm"
                            ]
                        ),

                    "representativeness":
                        float(
                            row[
                                "representativeness_norm"
                            ]
                        ),
                }
            )


        best = max(

            candidates,

            key=lambda item: (

                item[
                    "new_aspects"
                ],

                item[
                    "usefulness"
                ],

                item[
                    "representativeness"
                ],
            ),
        )


        best_idx = (
            best[
                "idx"
            ]
        )


        selected_indices.append(
            best_idx
        )


        covered_aspects |= (

            parse_pipe_set(

                product_df.loc[

                    best_idx,

                    "aspect_categories",
                ]
            )
        )


    return (

        product_df.loc[
            selected_indices
        ]

        .copy()

        .reset_index(
            drop=True
        )
    )


In [18]:
coverage_frames = []


for product_id, group in (

    experiment_df

    .groupby(
        "parent_asin"
    )
):

    selected = (

        select_aspect_coverage(

            product_df=group,

            budget=SELECT_K,
        )
    )


    coverage_frames.append(
        selected
    )


aspect_coverage_df = (

    pd.concat(

        coverage_frames,

        ignore_index=True,
    )
)


print(
    "Aspect-Coverage baseline rows:",
    len(
        aspect_coverage_df
    )
)


Aspect-Coverage baseline rows: 50


## 9. Parameterized Proposed Selector

The Notebook 05 selector is rewritten as a parameterized function so that the same implementation can be used for:

- Full Proposed,
- Proposed without Minority,
- ablation experiments,
- budget sensitivity,
- weight sensitivity.


In [19]:
def candidate_score_general(
    row,
    selected_embedding_rows,
    covered_aspects,
    total_product_aspects,
    covered_priority_groups,
    priority_groups,
    weights,
):

    candidate_aspects = (

        parse_pipe_set(

            row[
                "aspect_categories"
            ]
        )
    )


    candidate_minority_groups = (

        parse_pipe_set(

            row[
                "supported_minority_groups"
            ]
        )
    )


    new_aspects = (

        candidate_aspects

        - covered_aspects
    )


    uncovered_priority = (

        priority_groups

        - covered_priority_groups
    )


    new_priority_groups = (

        candidate_minority_groups

        & uncovered_priority
    )


    coverage_gain = (

        len(
            new_aspects
        )

        /

        max(

            len(
                total_product_aspects
            ),

            1,
        )
    )


    if priority_groups:

        minority_gain = (

            len(
                new_priority_groups
            )

            /

            len(
                priority_groups
            )
        )

    else:

        minority_gain = 0.0


    redundancy = (

        candidate_redundancy(

            row[
                "embedding_row"
            ],

            selected_embedding_rows,
        )
    )


    score = (

        weights[
            "usefulness"
        ]

        * float(
            row[
                "usefulness_norm"
            ]
        )


        + weights[
            "representativeness"
        ]

        * float(
            row[
                "representativeness_norm"
            ]
        )


        + weights[
            "coverage"
        ]

        * coverage_gain


        + weights[
            "minority"
        ]

        * minority_gain


        - weights[
            "redundancy"
        ]

        * redundancy
    )


    return {

        "selection_score":
            float(score),

        "coverage_gain":
            float(
                coverage_gain
            ),

        "minority_gain":
            float(
                minority_gain
            ),

        "redundancy":
            float(
                redundancy
            ),
    }


In [20]:
def select_generalized_proposed(
    product_df,
    budget,
    weights,
    priority_map,
    minority_reserved_slots,
):

    product_df = (

        product_df

        .copy()

        .reset_index(
            drop=True
        )
    )


    product_id = (

        product_df.loc[
            0,
            "parent_asin",
        ]
    )


    priority_groups = (

        priority_map.get(
            product_id,
            set(),
        )
    )


    total_product_aspects = set()


    for value in (

        product_df[
            "aspect_categories"
        ]
    ):

        total_product_aspects |= (

            parse_pipe_set(
                value
            )
        )


    selected_indices = []

    selected_embedding_rows = []

    covered_aspects = set()

    covered_priority_groups = set()


    def evaluate_candidate(
        idx
    ):

        row = product_df.loc[
            idx
        ]


        diagnostics = (

            candidate_score_general(

                row=row,

                selected_embedding_rows=
                    selected_embedding_rows,

                covered_aspects=
                    covered_aspects,

                total_product_aspects=
                    total_product_aspects,

                covered_priority_groups=
                    covered_priority_groups,

                priority_groups=
                    priority_groups,

                weights=
                    weights,
            )
        )


        return {

            "idx":
                idx,

            "tie_usefulness":
                float(
                    row[
                        "usefulness_norm"
                    ]
                ),

            **diagnostics,
        }


    reserve_limit = min(

        minority_reserved_slots,

        budget,
    )


    while (
        len(
            selected_indices
        )
        < reserve_limit
    ):

        uncovered_priority = (

            priority_groups

            - covered_priority_groups
        )


        if not uncovered_priority:

            break


        eligible = []


        for idx, row in (

            product_df.iterrows()
        ):

            if idx in selected_indices:

                continue


            candidate_groups = (

                parse_pipe_set(

                    row[
                        "supported_minority_groups"
                    ]
                )
            )


            if (

                candidate_groups

                & uncovered_priority
            ):

                eligible.append(

                    evaluate_candidate(
                        idx
                    )
                )


        if not eligible:

            break


        best = max(

            eligible,

            key=lambda item: (

                item[
                    "minority_gain"
                ],

                item[
                    "selection_score"
                ],

                item[
                    "coverage_gain"
                ],

                item[
                    "tie_usefulness"
                ],
            ),
        )


        best_idx = (
            best[
                "idx"
            ]
        )


        best_row = (
            product_df.loc[
                best_idx
            ]
        )


        selected_indices.append(
            best_idx
        )


        selected_embedding_rows.append(

            int(
                best_row[
                    "embedding_row"
                ]
            )
        )


        covered_aspects |= (

            parse_pipe_set(

                best_row[
                    "aspect_categories"
                ]
            )
        )


        covered_priority_groups |= (

            parse_pipe_set(

                best_row[
                    "supported_minority_groups"
                ]
            )

            & priority_groups
        )


    target_count = min(

        budget,

        len(
            product_df
        ),
    )


    while (
        len(
            selected_indices
        )
        < target_count
    ):

        candidates = []


        for idx, row in (

            product_df.iterrows()
        ):

            if idx in selected_indices:

                continue


            candidates.append(

                evaluate_candidate(
                    idx
                )
            )


        best = max(

            candidates,

            key=lambda item: (

                item[
                    "selection_score"
                ],

                item[
                    "minority_gain"
                ],

                item[
                    "coverage_gain"
                ],

                item[
                    "tie_usefulness"
                ],
            ),
        )


        best_idx = (
            best[
                "idx"
            ]
        )


        best_row = (
            product_df.loc[
                best_idx
            ]
        )


        selected_indices.append(
            best_idx
        )


        selected_embedding_rows.append(

            int(
                best_row[
                    "embedding_row"
                ]
            )
        )


        covered_aspects |= (

            parse_pipe_set(

                best_row[
                    "aspect_categories"
                ]
            )
        )


        covered_priority_groups |= (

            parse_pipe_set(

                best_row[
                    "supported_minority_groups"
                ]
            )

            & priority_groups
        )


    return (

        product_df.loc[
            selected_indices
        ]

        .copy()

        .reset_index(
            drop=True
        )
    )


In [21]:
def run_generalized_selector(
    selector_name,
    budget,
    weights,
    minority_reserved_slots,
):

    frames = []


    for product_id, group in (

        experiment_df

        .groupby(
            "parent_asin"
        )
    ):

        selected = (

            select_generalized_proposed(

                product_df=
                    group,

                budget=
                    budget,

                weights=
                    weights,

                priority_map=
                    priority_minority_by_product,

                minority_reserved_slots=
                    minority_reserved_slots,
            )
        )


        frames.append(
            selected
        )


    result_df = (

        pd.concat(
            frames,
            ignore_index=True,
        )
    )


    result_df[
        "selector_variant"
    ] = selector_name


    return result_df


## 10. Full Proposed and Proposed without Minority

The No-Minority variant removes:

- minority gain,
- reserved minority slots.

Other selector components remain unchanged.


In [22]:
full_proposed_df = (

    run_generalized_selector(

        selector_name=
            "FullProposed",

        budget=
            SELECT_K,

        weights=
            FULL_WEIGHTS,

        minority_reserved_slots=
            MINORITY_RESERVED_SLOTS,
    )
)


NO_MINORITY_WEIGHTS = {

    **FULL_WEIGHTS,

    "minority":
        0.0,
}


proposed_no_minority_df = (

    run_generalized_selector(

        selector_name=
            "ProposedNoMinority",

        budget=
            SELECT_K,

        weights=
            NO_MINORITY_WEIGHTS,

        minority_reserved_slots=
            0,
    )
)


print(
    "Full Proposed rows:",
    len(
        full_proposed_df
    )
)


print(
    "No-Minority rows:",
    len(
        proposed_no_minority_df
    )
)


Full Proposed rows: 50
No-Minority rows: 50


## 11. Validate Main Selector Outputs

All methods must use the same review budget and cover all products.


In [23]:
main_selection_sets = {

    "ObservedHelpfulTopK":
        observed_helpful_topk_df,

    "PredictedUsefulnessTopK":
        predicted_usefulness_topk_df,

    "HelpfulnessMMR":
        helpfulness_mmr_df,

    "AspectCoverage":
        aspect_coverage_df,

    "ProposedNoMinority":
        proposed_no_minority_df,

    "FullProposed":
        full_proposed_df,
}


for name, frame in (

    main_selection_sets.items()
):

    assert frame[
        "review_id"
    ].is_unique


    assert (

        frame[
            "parent_asin"
        ].nunique()

        ==

        experiment_df[
            "parent_asin"
        ].nunique()
    )


    counts = (

        frame

        .groupby(
            "parent_asin"
        )

        .size()
    )


    assert (
        counts
        == SELECT_K
    ).all()


    print(
        f"PASS - {name}:",
        len(frame),
        "rows"
    )


print(
    "\nAll main selector outputs validated."
)


PASS - ObservedHelpfulTopK: 50 rows
PASS - PredictedUsefulnessTopK: 50 rows
PASS - HelpfulnessMMR: 50 rows
PASS - AspectCoverage: 50 rows
PASS - ProposedNoMinority: 50 rows
PASS - FullProposed: 50 rows

All main selector outputs validated.


### Baseline Selection Audit

The exact review IDs selected by each main baseline are retained for traceability and later audit.


In [24]:
selection_audit_frames = []

for selector_name, frame in (
    main_selection_sets.items()
):

    temp = frame.copy()

    temp[
        "selector"
    ] = selector_name

    selection_audit_frames.append(
        temp
    )


baseline_selected_reviews_df = (
    pd.concat(
        selection_audit_frames,
        ignore_index=True,
    )
)


print(
    "Baseline selection audit rows:",
    len(
        baseline_selected_reviews_df
    )
)

display(
    baseline_selected_reviews_df[
        [
            "selector",
            "parent_asin",
            "review_id",
        ]
    ].head(20)
)


Baseline selection audit rows: 300


,selector,parent_asin,review_id
0,ObservedHelpfulTopK,B07GYY6VTD,6f8a74cde06a1711
1,ObservedHelpfulTopK,B07GYY6VTD,b068d451b1f068c1
2,ObservedHelpfulTopK,B07GYY6VTD,7f299a542d0751d5
3,ObservedHelpfulTopK,B07GYY6VTD,8455e305638bffcc
4,ObservedHelpfulTopK,B07GYY6VTD,5aac94ce974cf360
5,ObservedHelpfulTopK,B07SDQ483V,8ea0f6f697d7447e
6,ObservedHelpfulTopK,B07SDQ483V,e59a667ea378af3a
7,ObservedHelpfulTopK,B07SDQ483V,48ed4157c0fc7f9e
8,ObservedHelpfulTopK,B07SDQ483V,0bc8961338f7f912
9,ObservedHelpfulTopK,B07SDQ483V,16e33c097e8fee6f


## 12. Unified Baseline Comparison

All methods are evaluated using the same product-level evaluation metrics.


In [25]:
main_metric_frames = []


for selector_name, frame in (

    main_selection_sets.items()
):

    metrics = (

        summarize_selection(

            frame,

            selector_name,
        )
    )


    main_metric_frames.append(
        metrics
    )


main_comparison_product_df = (

    pd.concat(

        main_metric_frames,

        ignore_index=True,
    )
)


main_comparison_overall_df = (

    overall_summary(

        main_comparison_product_df
    )
)


display(
    main_comparison_overall_df
)


,selector,mean_usefulness_norm,mean_representativeness_norm,mean_observed_log_helpful,mean_aspect_coverage,mean_supported_minority_recall,mean_priority_minority_coverage,mean_pairwise_similarity,mean_opinion_js_divergence
0,AspectCoverage,0.851174,0.712649,2.206056,1.000000,0.306072,0.15,0.475217,0.277468
1,FullProposed,0.742725,0.766597,1.824493,0.933333,0.416744,1.00,0.492611,0.336771
2,HelpfulnessMMR,0.850566,0.691175,2.124496,0.800000,0.293057,0.10,0.438294,0.332131
3,ObservedHelpfulTopK,0.768678,0.681043,2.787884,0.866667,0.322098,0.20,0.411797,0.314594
4,PredictedUsefulnessTopK,0.859078,0.711964,2.222749,0.816667,0.278294,0.10,0.472905,0.316286
5,ProposedNoMinority,0.834591,0.772256,2.199952,0.966667,0.281072,0.20,0.510038,0.296061


## 13. Ablation Study

The following proposed-selector components are removed one at a time:

- minority preservation,
- redundancy control,
- coverage,
- representativeness.

The full proposed selector is retained as the reference.


In [26]:
ablation_configs = {

    "FullProposed": {

        "weights":
            FULL_WEIGHTS,

        "reserved_slots":
            MINORITY_RESERVED_SLOTS,
    },


    "NoMinority": {

        "weights": {

            **FULL_WEIGHTS,

            "minority":
                0.0,
        },

        "reserved_slots":
            0,
    },


    "NoRedundancy": {

        "weights": {

            **FULL_WEIGHTS,

            "redundancy":
                0.0,
        },

        "reserved_slots":
            MINORITY_RESERVED_SLOTS,
    },


    "NoCoverage": {

        "weights": {

            **FULL_WEIGHTS,

            "coverage":
                0.0,
        },

        "reserved_slots":
            MINORITY_RESERVED_SLOTS,
    },


    "NoRepresentativeness": {

        "weights": {

            **FULL_WEIGHTS,

            "representativeness":
                0.0,
        },

        "reserved_slots":
            MINORITY_RESERVED_SLOTS,
    },
}


In [27]:
ablation_metric_frames = []


for variant_name, config in (

    ablation_configs.items()
):

    selection = (

        run_generalized_selector(

            selector_name=
                variant_name,

            budget=
                SELECT_K,

            weights=
                config[
                    "weights"
                ],

            minority_reserved_slots=
                config[
                    "reserved_slots"
                ],
        )
    )


    metrics = (

        summarize_selection(

            selection,

            variant_name,
        )
    )


    ablation_metric_frames.append(
        metrics
    )


ablation_product_df = (

    pd.concat(

        ablation_metric_frames,

        ignore_index=True,
    )
)


ablation_overall_df = (

    overall_summary(

        ablation_product_df
    )
)


display(
    ablation_overall_df
)


,selector,mean_usefulness_norm,mean_representativeness_norm,mean_observed_log_helpful,mean_aspect_coverage,mean_supported_minority_recall,mean_priority_minority_coverage,mean_pairwise_similarity,mean_opinion_js_divergence
0,FullProposed,0.742725,0.766597,1.824493,0.933333,0.416744,1.0,0.492611,0.336771
1,NoCoverage,0.750342,0.772539,1.989514,0.833333,0.408411,1.0,0.501894,0.383803
2,NoMinority,0.834591,0.772256,2.199952,0.966667,0.281072,0.2,0.510038,0.296061
3,NoRedundancy,0.728990,0.792706,1.869119,0.950000,0.442855,1.0,0.514537,0.331134
4,NoRepresentativeness,0.736532,0.647577,1.810330,0.933333,0.398376,1.0,0.377265,0.334438


## 14. Review-Budget Sensitivity

The full proposed selector is tested using:

- k = 3
- k = 5
- k = 7
- k = 10


In [28]:
BUDGET_VALUES = [
    3,
    5,
    7,
    10,
]


budget_frames = []


for budget_value in (
    BUDGET_VALUES
):

    selection = (

        run_generalized_selector(

            selector_name=
                f"FullProposed_k{budget_value}",

            budget=
                budget_value,

            weights=
                FULL_WEIGHTS,

            minority_reserved_slots=
                min(
                    MINORITY_RESERVED_SLOTS,
                    budget_value,
                ),
        )
    )


    metrics = (

        summarize_selection(

            selection,

            f"FullProposed_k{budget_value}",
        )
    )


    metrics[
        "budget_k"
    ] = budget_value


    budget_frames.append(
        metrics
    )


budget_sensitivity_product_df = (

    pd.concat(

        budget_frames,

        ignore_index=True,
    )
)


budget_sensitivity_overall_df = (

    budget_sensitivity_product_df

    .groupby(
        "budget_k",
        as_index=False,
    )

    .agg(

        mean_usefulness_norm=(
            "mean_usefulness_norm",
            "mean",
        ),

        mean_representativeness_norm=(
            "mean_representativeness_norm",
            "mean",
        ),

        mean_aspect_coverage=(
            "aspect_coverage",
            "mean",
        ),

        mean_supported_minority_recall=(
            "supported_minority_recall",
            "mean",
        ),

        mean_priority_minority_coverage=(
            "priority_minority_coverage",
            "mean",
        ),

        mean_pairwise_similarity=(
            "mean_pairwise_similarity",
            "mean",
        ),

        mean_opinion_js_divergence=(
            "opinion_js_divergence",
            "mean",
        ),
    )
)


display(
    budget_sensitivity_overall_df
)


,budget_k,mean_usefulness_norm,mean_representativeness_norm,mean_aspect_coverage,mean_supported_minority_recall,mean_priority_minority_coverage,mean_pairwise_similarity,mean_opinion_js_divergence
0,3,0.671043,0.783232,0.716667,0.327183,1.0,0.477134,0.522149
1,5,0.742725,0.766597,0.933333,0.416744,1.0,0.492611,0.336771
2,7,0.738860,0.779718,0.983333,0.482148,1.0,0.503333,0.264055
3,10,0.717390,0.775821,1.000000,0.677615,1.0,0.497510,0.158290


## 15. Minority-Weight Sensitivity

The minority-preservation weight is varied while other weights remain fixed.

The minority-reservation mechanism is held constant so that this experiment isolates the effect of the minority score weight. Complete removal of the minority mechanism is evaluated separately in the No-Minority ablation.


In [29]:
MINORITY_WEIGHT_VALUES = [
    0.0,
    0.10,
    0.25,
    0.40,
]


minority_weight_frames = []


for minority_weight in (

    MINORITY_WEIGHT_VALUES
):

    weights = {

        **FULL_WEIGHTS,

        "minority":
            minority_weight,
    }


    # Keep the minority-reservation mechanism constant so
    # this experiment isolates only the minority score weight.
    reserved_slots = MINORITY_RESERVED_SLOTS


    selection = (

        run_generalized_selector(

            selector_name=
                f"MinorityWeight_{minority_weight}",

            budget=
                SELECT_K,

            weights=
                weights,

            minority_reserved_slots=
                reserved_slots,
        )
    )


    metrics = (

        summarize_selection(

            selection,

            f"MinorityWeight_{minority_weight}",
        )
    )


    metrics[
        "minority_weight"
    ] = minority_weight


    minority_weight_frames.append(
        metrics
    )


minority_weight_product_df = (

    pd.concat(

        minority_weight_frames,

        ignore_index=True,
    )
)


minority_weight_overall_df = (

    minority_weight_product_df

    .groupby(
        "minority_weight",
        as_index=False,
    )

    .agg(

        mean_usefulness_norm=(
            "mean_usefulness_norm",
            "mean",
        ),

        mean_supported_minority_recall=(
            "supported_minority_recall",
            "mean",
        ),

        mean_priority_minority_coverage=(
            "priority_minority_coverage",
            "mean",
        ),

        mean_pairwise_similarity=(
            "mean_pairwise_similarity",
            "mean",
        ),

        mean_opinion_js_divergence=(
            "opinion_js_divergence",
            "mean",
        ),
    )
)


display(
    minority_weight_overall_df
)


,minority_weight,mean_usefulness_norm,mean_supported_minority_recall,mean_priority_minority_coverage,mean_pairwise_similarity,mean_opinion_js_divergence
0,0.00,0.742725,0.416744,1.0,0.492611,0.336771
1,0.10,0.742725,0.416744,1.0,0.492611,0.336771
2,0.25,0.742725,0.416744,1.0,0.492611,0.336771
3,0.40,0.742725,0.416744,1.0,0.492611,0.336771


## 16. Redundancy-Weight Sensitivity

The redundancy penalty is varied while the remaining selector settings stay fixed.


In [30]:
REDUNDANCY_WEIGHT_VALUES = [
    0.0,
    0.10,
    0.20,
    0.30,
    0.40,
]


redundancy_weight_frames = []


for redundancy_weight in (

    REDUNDANCY_WEIGHT_VALUES
):

    weights = {

        **FULL_WEIGHTS,

        "redundancy":
            redundancy_weight,
    }


    selection = (

        run_generalized_selector(

            selector_name=
                f"RedundancyWeight_{redundancy_weight}",

            budget=
                SELECT_K,

            weights=
                weights,

            minority_reserved_slots=
                MINORITY_RESERVED_SLOTS,
        )
    )


    metrics = (

        summarize_selection(

            selection,

            f"RedundancyWeight_{redundancy_weight}",
        )
    )


    metrics[
        "redundancy_weight"
    ] = redundancy_weight


    redundancy_weight_frames.append(
        metrics
    )


redundancy_weight_product_df = (

    pd.concat(

        redundancy_weight_frames,

        ignore_index=True,
    )
)


redundancy_weight_overall_df = (

    redundancy_weight_product_df

    .groupby(
        "redundancy_weight",
        as_index=False,
    )

    .agg(

        mean_usefulness_norm=(
            "mean_usefulness_norm",
            "mean",
        ),

        mean_representativeness_norm=(
            "mean_representativeness_norm",
            "mean",
        ),

        mean_supported_minority_recall=(
            "supported_minority_recall",
            "mean",
        ),

        mean_pairwise_similarity=(
            "mean_pairwise_similarity",
            "mean",
        ),

        mean_opinion_js_divergence=(
            "opinion_js_divergence",
            "mean",
        ),
    )
)


display(
    redundancy_weight_overall_df
)


,redundancy_weight,mean_usefulness_norm,mean_representativeness_norm,mean_supported_minority_recall,mean_pairwise_similarity,mean_opinion_js_divergence
0,0.0,0.728990,0.792706,0.442855,0.514537,0.331134
1,0.1,0.733350,0.783767,0.441744,0.510159,0.332804
2,0.2,0.742725,0.766597,0.416744,0.492611,0.336771
3,0.3,0.742218,0.760940,0.417855,0.480449,0.340288
4,0.4,0.736264,0.743530,0.436659,0.451216,0.340238


## 17. Notebook 05 Replication Check

The generalized Full Proposed selector should reproduce the Notebook 05 prototype under the same configuration.


In [31]:
saved_ids = set(

    saved_full_proposed_df[
        "review_id"
    ]
)


recomputed_ids = set(

    full_proposed_df[
        "review_id"
    ]
)


intersection = len(

    saved_ids
    & recomputed_ids
)


union = len(

    saved_ids
    | recomputed_ids
)


jaccard_overlap = (

    intersection
    / union

    if union > 0

    else 1.0
)


print(
    "Saved IDs:",
    len(
        saved_ids
    )
)


print(
    "Recomputed IDs:",
    len(
        recomputed_ids
    )
)


print(
    "Exact set match:",
    saved_ids
    == recomputed_ids
)


print(
    "Jaccard overlap:",
    round(
        jaccard_overlap,
        4,
    )
)

assert (
    saved_ids
    == recomputed_ids
), (
    "Replication failed: the generalized Full Proposed "
    "selector does not reproduce Notebook 05 selections."
)

print(
    "PASS - Notebook 05 Full Proposed selections reproduced."
)


Saved IDs: 50
Recomputed IDs: 50
Exact set match: True
Jaccard overlap: 1.0
PASS - Notebook 05 Full Proposed selections reproduced.


## 18. Save Evaluation Outputs

Compact result tables are saved for reproducibility and PP1 evidence.


In [32]:
MAIN_PRODUCT_PATH = (

    EVALUATION_DIR
    / "baseline_comparison_by_product.csv"
)


MAIN_OVERALL_PATH = (

    EVALUATION_DIR
    / "baseline_comparison_overall.csv"
)


ABLATION_PRODUCT_PATH = (

    EVALUATION_DIR
    / "ablation_by_product.csv"
)


ABLATION_OVERALL_PATH = (

    EVALUATION_DIR
    / "ablation_overall.csv"
)


BUDGET_PRODUCT_PATH = (

    EVALUATION_DIR
    / "budget_sensitivity_by_product.csv"
)


BUDGET_OVERALL_PATH = (

    EVALUATION_DIR
    / "budget_sensitivity_overall.csv"
)


MINORITY_WEIGHT_PATH = (

    EVALUATION_DIR
    / "minority_weight_sensitivity.csv"
)


MINORITY_WEIGHT_PRODUCT_PATH = (

    EVALUATION_DIR
    / "minority_weight_sensitivity_by_product.csv"
)


REDUNDANCY_WEIGHT_PATH = (

    EVALUATION_DIR
    / "redundancy_weight_sensitivity.csv"
)


REDUNDANCY_WEIGHT_PRODUCT_PATH = (

    EVALUATION_DIR
    / "redundancy_weight_sensitivity_by_product.csv"
)




BASELINE_SELECTIONS_PATH = (

    EVALUATION_DIR
    / "baseline_selected_reviews.csv"
)

CONFIG_PATH = (

    EVALUATION_DIR
    / "evaluation_config.json"
)


In [33]:
main_comparison_product_df.to_csv(

    MAIN_PRODUCT_PATH,

    index=False,
)


main_comparison_overall_df.to_csv(

    MAIN_OVERALL_PATH,

    index=False,
)


ablation_product_df.to_csv(

    ABLATION_PRODUCT_PATH,

    index=False,
)


ablation_overall_df.to_csv(

    ABLATION_OVERALL_PATH,

    index=False,
)


budget_sensitivity_product_df.to_csv(

    BUDGET_PRODUCT_PATH,

    index=False,
)


budget_sensitivity_overall_df.to_csv(

    BUDGET_OVERALL_PATH,

    index=False,
)


minority_weight_overall_df.to_csv(

    MINORITY_WEIGHT_PATH,

    index=False,
)


minority_weight_product_df.to_csv(

    MINORITY_WEIGHT_PRODUCT_PATH,

    index=False,
)


redundancy_weight_overall_df.to_csv(

    REDUNDANCY_WEIGHT_PATH,

    index=False,
)


redundancy_weight_product_df.to_csv(

    REDUNDANCY_WEIGHT_PRODUCT_PATH,

    index=False,
)


baseline_selected_reviews_df.to_csv(

    BASELINE_SELECTIONS_PATH,

    index=False,
)


evaluation_config = {

    "budget_k":
        SELECT_K,

    "mmr_lambda":
        MMR_LAMBDA,

    "full_weights":
        FULL_WEIGHTS,

    "minority_reserved_slots":
        MINORITY_RESERVED_SLOTS,

    "budget_values":
        BUDGET_VALUES,

    "minority_weight_values":
        MINORITY_WEIGHT_VALUES,

    "redundancy_weight_values":
        REDUNDANCY_WEIGHT_VALUES,

    "baselines": [

        "ObservedHelpfulTopK",

        "PredictedUsefulnessTopK",

        "HelpfulnessMMR",

        "AspectCoverage",

        "ProposedNoMinority",

        "FullProposed",
    ],

    "mock_c1_warning":
        (
            "Aspect/sentiment, minority, coverage "
            "and fidelity outputs use synthetic "
            "integration-test C1 labels."
        ),
}


with open(

    CONFIG_PATH,

    "w",

    encoding="utf-8",

) as f:

    json.dump(

        evaluation_config,

        f,

        indent=2,
    )


print(
    "Saved:",
    MAIN_PRODUCT_PATH
)

print(
    "Saved:",
    MAIN_OVERALL_PATH
)

print(
    "Saved:",
    ABLATION_PRODUCT_PATH
)

print(
    "Saved:",
    ABLATION_OVERALL_PATH
)

print(
    "Saved:",
    BUDGET_PRODUCT_PATH
)

print(
    "Saved:",
    BUDGET_OVERALL_PATH
)

print(
    "Saved:",
    MINORITY_WEIGHT_PATH
)


print(
    "Saved:",
    MINORITY_WEIGHT_PRODUCT_PATH
)

print(
    "Saved:",
    REDUNDANCY_WEIGHT_PATH
)


print(
    "Saved:",
    REDUNDANCY_WEIGHT_PRODUCT_PATH
)

print(
    "Saved:",
    CONFIG_PATH
)


print(
    "Saved:",
    BASELINE_SELECTIONS_PATH
)


Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\baseline_comparison_by_product.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\baseline_comparison_overall.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\ablation_by_product.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\ablation_overall.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\budget_sensitivity_by_product.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\budget_sensitivity_overall.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\minority_weight_sensitivity.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\minority_weight_sensitivity_by_product.csv
Saved: d:\Research\Repository\J26-DS-349\component_2\results\selector_evaluation\redundancy_weight_s

## 19. Final Evaluation Validation

The following checks verify that:

- all baseline selectors respect the same budget,
- all products are represented,
- ablation results were generated,
- sensitivity experiments were generated,
- all expected evidence files were saved.


In [34]:
expected_rows = (

    SELECT_K

    * experiment_df[
        "parent_asin"
    ].nunique()
)


for name, frame in (

    main_selection_sets.items()
):

    assert (
        len(frame)
        == expected_rows
    )


assert (

    main_comparison_product_df[
        "parent_asin"
    ].nunique()

    ==

    experiment_df[
        "parent_asin"
    ].nunique()
)


assert (

    len(
        budget_sensitivity_overall_df
    )

    ==

    len(
        BUDGET_VALUES
    )
)


assert (

    len(
        minority_weight_overall_df
    )

    ==

    len(
        MINORITY_WEIGHT_VALUES
    )
)


assert (

    len(
        redundancy_weight_overall_df
    )

    ==

    len(
        REDUNDANCY_WEIGHT_VALUES
    )
)


for path in [

    MAIN_PRODUCT_PATH,

    MAIN_OVERALL_PATH,

    ABLATION_PRODUCT_PATH,

    ABLATION_OVERALL_PATH,

    BUDGET_PRODUCT_PATH,

    BUDGET_OVERALL_PATH,

    MINORITY_WEIGHT_PATH,

    MINORITY_WEIGHT_PRODUCT_PATH,

    REDUNDANCY_WEIGHT_PATH,

    REDUNDANCY_WEIGHT_PRODUCT_PATH,

    BASELINE_SELECTIONS_PATH,

    CONFIG_PATH,
]:

    assert path.exists()


print(
    "Baseline, ablation and sensitivity "
    "evaluation validation passed."
)


Baseline, ablation and sensitivity evaluation validation passed.


## 20. Evaluation Milestone Summary

### Implemented Baselines

- Observed Helpful-Vote Top-K
- Predicted-Usefulness Top-K
- Helpfulness + MMR
- Aspect-Coverage
- Proposed without Minority
- Full Proposed

### Implemented Ablations

- No Minority
- No Redundancy
- No Coverage
- No Representativeness

### Implemented Sensitivity Experiments

- Review-budget sensitivity
- Minority-weight sensitivity
- Redundancy-weight sensitivity

### Current Limitations

- Final C1 aspect/sentiment outputs are not yet available.
- Current minority, coverage and opinion-fidelity metrics use synthetic integration-test labels.
- Current experiments use the smartphone development pilot.
- Function 1 remains the preliminary TF-IDF + Ridge usefulness baseline.
- Final unbiased evaluation and statistical validation remain later research work.

### Status

**Selector baseline, ablation and sensitivity evaluation implemented and ready for execution testing.**
